# Electricity Demand Forecasting Pipeline
**Target:** Predict next hour's `demand_mw` using classical ML  
**Models:** LightGBM (primary), XGBoost, Random Forest  
**Metric:** MAPE (primary), RMSE (secondary)

## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgbm

RANDOM_STATE = 42
SPLIT_DATE   = '2023-01-01'
TARGET_COL   = 'demand_mw'

## Step 1 — Load Data

In [ ]:
demand = pd.read_excel('data/PGCB_date_power_demand.xlsx')
weather = pd.read_excel('data/weather_data.xlsx')
econ    = pd.read_csv('data/economic_full_1.csv')

print('Demand shape :', demand.shape)
print('Weather shape:', weather.shape)
print('Econ shape   :', econ.shape)
print('\nDemand columns:', demand.columns.tolist())
print('Weather columns:', weather.columns.tolist())
print('Econ columns   :', econ.columns.tolist())

## Step 2 — Clean & Align Timestamps

In [ ]:
# ── 2a. Parse datetimes ──────────────────────────────────────────────────────
demand['datetime']  = pd.to_datetime(demand['datetime'],  errors='coerce')
weather['datetime'] = pd.to_datetime(weather['datetime'], errors='coerce')

# drop rows where datetime itself failed to parse
demand  = demand.dropna(subset=['datetime'])
weather = weather.dropna(subset=['datetime'])

# ── 2b. Remove duplicates ────────────────────────────────────────────────────
before = len(demand)
demand  = demand.drop_duplicates(subset='datetime')
weather = weather.drop_duplicates(subset='datetime')
print(f'Demand duplicates removed: {before - len(demand)}')

# ── 2c. Merge weather on datetime (left join keeps all demand rows) ───────────
df = demand.merge(weather, on='datetime', how='left')

# ── 2d. Set index, sort, resample to strict hourly grid ──────────────────────
df = df.set_index('datetime').sort_index()
df = df.resample('H').mean()          # fills gaps, averages any sub-hourly noise

print(f'\nDate range : {df.index.min()} → {df.index.max()}')
print(f'Total hours: {len(df)}')
print(f'Missing demand_mw before fill: {df[TARGET_COL].isna().sum()}')

In [ ]:
# ── 2e. Tiered missing value strategy ────────────────────────────────────────
# Gaps ≤ 3 h  → linear interpolation (sensor dropout)
# Gaps  > 3 h → fill from same hour the previous day (structural outage)

def tiered_fill(series, short_gap=3):
    s = series.copy()
    # mark NaN runs
    mask = s.isna()
    run_id = (mask != mask.shift()).cumsum()
    run_len = mask.groupby(run_id).transform('sum')

    # short gaps → interpolate
    short_mask = mask & (run_len <= short_gap)
    s[short_mask] = s.interpolate(method='time')[short_mask]

    # long gaps → same hour previous day
    still_nan = s.isna()
    s[still_nan] = s.shift(24)[still_nan]

    # last resort: forward fill any remaining
    s = s.ffill().bfill()
    return s

df[TARGET_COL] = tiered_fill(df[TARGET_COL])

# Fill weather gaps with simple interpolation
weather_cols = [c for c in df.columns if c != TARGET_COL]
df[weather_cols] = df[weather_cols].interpolate(method='time').ffill().bfill()

print(f'Missing demand_mw after fill: {df[TARGET_COL].isna().sum()}')

## Step 3 — Anomaly Detection & Smoothing

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(df[TARGET_COL], linewidth=0.5, color='steelblue')
axes[0].set_title('demand_mw — BEFORE anomaly removal')

# ── Rolling median ± 3σ clipping ─────────────────────────────────────────────
# center=True gives symmetric window (safe since we're on training prep, not live)
roll_med = df[TARGET_COL].rolling(window=24*7, center=True, min_periods=12).median()
roll_std = df[TARGET_COL].rolling(window=24*7, center=True, min_periods=12).std()

upper = roll_med + 3 * roll_std
lower = roll_med - 3 * roll_std

# clip (not delete) — preserves the hourly index
df[TARGET_COL] = df[TARGET_COL].clip(lower=lower, upper=upper)

# interpolate any NaNs that clipping introduced at series edges
df[TARGET_COL] = df[TARGET_COL].interpolate(method='time').ffill().bfill()

axes[1].plot(df[TARGET_COL], linewidth=0.5, color='darkorange')
axes[1].set_title('demand_mw — AFTER anomaly removal')
plt.tight_layout()
plt.savefig('outputs/anomaly_check.png', dpi=150)
plt.show()
print('Anomaly clipping done.')

## Step 4 — Economic Data Integration

In [ ]:
# ── Select only demand-relevant macro columns ────────────────────────────────
# Adjust column names to match your actual CSV headers
ECON_COLS = ['year', 'gdp_growth', 'industrial_output', 'population']
available_econ = [c for c in ECON_COLS if c in econ.columns]
econ = econ[available_econ].copy()
econ['year'] = econ['year'].astype(int)

# ── Join on calendar year ────────────────────────────────────────────────────
df['year'] = df.index.year
df = df.merge(econ, on='year', how='left')
df = df.set_index(df.index)  # ensure DatetimeIndex is intact after merge

print('Econ columns added:', [c for c in econ.columns if c != 'year'])
print('Missing econ values:', df[available_econ[1:]].isna().sum().to_dict())

## Step 5 — Feature Engineering

In [ ]:
# ── 5a. Calendar / time features ─────────────────────────────────────────────
df['hour']       = df.index.hour
df['dayofweek']  = df.index.dayofweek
df['month']      = df.index.month
df['quarter']    = df.index.quarter
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

# cyclic encoding — prevents 23→0 discontinuity
df['hour_sin']   = np.sin(2 * np.pi * df['hour']  / 24)
df['hour_cos']   = np.cos(2 * np.pi * df['hour']  / 24)
df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin']    = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']    = np.cos(2 * np.pi * df['dayofweek'] / 7)

print('Calendar features done.')

In [ ]:
# ── 5b. Lag features ──────────────────────────────────────────────────────────
# All lags use past values only → no leakage
LAGS = [1, 2, 3, 6, 12, 24, 48, 168]
for lag in LAGS:
    df[f'lag_{lag}'] = df[TARGET_COL].shift(lag)

print(f'Lag features added: {[f"lag_{l}" for l in LAGS]}')

In [ ]:
# ── 5c. Rolling statistics ────────────────────────────────────────────────────
# .shift(1) BEFORE .rolling() → uses data strictly before current row
base = df[TARGET_COL].shift(1)

df['rolling_mean_3h']    = base.rolling(3).mean()
df['rolling_mean_24h']   = base.rolling(24).mean()
df['rolling_std_24h']    = base.rolling(24).std()
df['rolling_mean_168h']  = base.rolling(168).mean()   # weekly baseline
df['rolling_max_24h']    = base.rolling(24).max()
df['rolling_min_24h']    = base.rolling(24).min()

print('Rolling features done.')

In [ ]:
# ── 5d. Weather + interaction features ───────────────────────────────────────
# Adjust 'temperature'/'humidity' to your actual column names
if 'temperature' in df.columns:
    df['cooling_demand']  = np.maximum(0, df['temperature'] - 24)   # AC load
    df['heating_demand']  = np.maximum(0, 18 - df['temperature'])   # heating load
    df['temp_change']     = df['temperature'].diff()                 # rate of change
    if 'humidity' in df.columns:
        df['heat_index'] = df['temperature'] * df['humidity']        # perceived heat

print('Weather features done.')

## Step 6 — Target + Final Cleanup

In [ ]:
# next-hour demand is the prediction target
df['target'] = df[TARGET_COL].shift(-1)

# drop rows where ANY critical feature or target is NaN
critical_cols = [f'lag_{l}' for l in LAGS] + ['rolling_mean_24h', 'target']
before = len(df)
df = df.dropna(subset=critical_cols)
print(f'Rows dropped (NaN cleanup): {before - len(df)}')
print(f'Final dataset shape: {df.shape}')
print(f'Date range: {df.index.min()} → {df.index.max()}')

## Step 7 — Train / Test Split

In [ ]:
# strictly chronological — no shuffle, no random state needed here
DROP_COLS = ['target', TARGET_COL, 'year']   # drop raw demand & leaky cols from X

train = df[df.index < SPLIT_DATE]
test  = df[df.index >= SPLIT_DATE]

feature_cols = [c for c in df.columns if c not in DROP_COLS]

X_train, y_train = train[feature_cols], train['target']
X_test,  y_test  = test[feature_cols],  test['target']

print(f'Train: {len(train)} rows  |  {train.index.min().date()} → {train.index.max().date()}')
print(f'Test : {len(test)}  rows  |  {test.index.min().date()}  → {test.index.max().date()}')
print(f'Features: {len(feature_cols)}')

## Step 8 — Metric Definition

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — primary metric for this task."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # guard against zero-division on near-zero demand rows
    mask = y_true > 1
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate(name, y_true, y_pred):
    m = mape(y_true, y_pred)
    r = rmse(y_true, y_pred)
    print(f'{name:30s}  MAPE={m:.3f}%   RMSE={r:.2f} MW')
    return m, r

## Step 9 — Naive Baseline (must beat this)

In [ ]:
# Baseline: predict next hour = current hour (lag_1)
baseline_pred = X_test['lag_1'].values
baseline_mape, baseline_rmse = evaluate('Baseline (lag_1)', y_test, baseline_pred)
print('\n→ Any model MUST beat this. If not, pipeline has a bug.')

## Step 10 — TimeSeriesSplit Cross-Validation + LightGBM

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

# Use a lightweight model for CV to find rough hyperparams
lgb_cv = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbose=-1
)

print('TimeSeriesSplit CV (5 folds):')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    X_tr,  X_val  = X_train.iloc[tr_idx],  X_train.iloc[val_idx]
    y_tr,  y_val  = y_train.iloc[tr_idx],  y_train.iloc[val_idx]

    lgb_cv.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgbm.early_stopping(stopping_rounds=50, verbose=False),
            lgbm.log_evaluation(period=-1)
        ]
    )
    fold_mape = mape(y_val, lgb_cv.predict(X_val))
    cv_scores.append(fold_mape)
    print(f'  Fold {fold}: MAPE = {fold_mape:.3f}%')

print(f'\nMean CV MAPE: {np.mean(cv_scores):.3f}% ± {np.std(cv_scores):.3f}%')

## Step 11 — Final Model Training

In [ ]:
# ── LightGBM — primary model ──────────────────────────────────────────────────
lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbose=-1
)

# Use last 10% of training as internal val for early stopping
val_cutoff = int(len(X_train) * 0.9)
X_tr_final, X_val_final = X_train.iloc[:val_cutoff], X_train.iloc[val_cutoff:]
y_tr_final, y_val_final = y_train.iloc[:val_cutoff], y_train.iloc[val_cutoff:]

lgb_model.fit(
    X_tr_final, y_tr_final,
    eval_set=[(X_val_final, y_val_final)],
    callbacks=[
        lgbm.early_stopping(stopping_rounds=50, verbose=True),
        lgbm.log_evaluation(period=100)
    ]
)
print(f'Best iteration: {lgb_model.best_iteration_}')

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_model.fit(X_train, y_train)

print('All models trained.')

## Step 12 — Evaluation on Test Set

In [ ]:
lgb_pred = lgb_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)
rf_pred  = rf_model.predict(X_test)

print('\n=== FINAL TEST SET RESULTS ===')
evaluate('Baseline (lag_1)',   y_test, baseline_pred)
lgb_mape, lgb_rmse = evaluate('LightGBM',  y_test, lgb_pred)
evaluate('XGBoost',            y_test, xgb_pred)
evaluate('Random Forest',      y_test, rf_pred)

# ── Sanity check on predictions ───────────────────────────────────────────────
hist_max = df[TARGET_COL].max()
assert (lgb_pred > 0).all(),                  'ERROR: negative predictions found'
assert (lgb_pred < hist_max * 1.5).all(),     'ERROR: predictions exceed 1.5× historical max'
print(f'\n✓ Sanity check passed  (pred range: {lgb_pred.min():.0f}–{lgb_pred.max():.0f} MW)')

In [ ]:
# ── Actual vs Predicted plot (first 2 weeks of test set) ─────────────────────
plot_n = 24 * 14
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y_test.values[:plot_n],   label='Actual',    linewidth=1.2)
ax.plot(lgb_pred[:plot_n],        label='LightGBM',  linewidth=1.0, alpha=0.85)
ax.set_title('Actual vs Predicted — First 2 Weeks of 2023')
ax.set_ylabel('Demand (MW)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/predictions_vs_actual.png', dpi=150)
plt.show()

In [ ]:
# ── Save predictions ──────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'datetime':    test.index,
    'actual':      y_test.values,
    'lgb_pred':    lgb_pred,
    'xgb_pred':    xgb_pred,
    'rf_pred':     rf_pred,
    'baseline':    baseline_pred
})
results_df.to_csv('outputs/predictions.csv', index=False)
print('Predictions saved → outputs/predictions.csv')

## Step 13 — Feature Importance (Required Deliverable)

In [ ]:
feat_imp = pd.Series(
    lgb_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(9, 7))
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('LightGBM — Top 20 Feature Importances', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150)
plt.show()

print('\nTop 10 features:')
print(feat_imp.tail(10).sort_values(ascending=False).to_string())

## Step 14 — Summary Report

### A. Data Preparation & Anomaly Handling
- Removed duplicate timestamps in PGCB and weather datasets.
- Resampled to a strict hourly grid using `.resample('H').mean()` to eliminate irregular intervals.
- **Tiered missing value strategy:** gaps ≤ 3 h were filled with linear time interpolation (sensor dropouts); longer gaps were filled using the same hour from the previous day (structural outages).
- **Anomaly detection:** applied rolling 7-day median ± 3σ clipping to `demand_mw`. Clipping (not deletion) was used to preserve the continuous hourly index required for lag features. Edge NaNs after clipping were resolved with `interpolate(method='time')`.

### B. Feature Engineering Rationale
| Feature Group | Features | Why |
|---|---|---|
| Calendar | hour, dow, month, is_weekend | Demand follows strong daily & weekly cycles |
| Cyclic encoding | hour_sin/cos, month_sin/cos, dow_sin/cos | Prevents discontinuity at period boundaries (e.g. hour 23→0) |
| Lag features | lag_1,2,3,6,12,24,48,168 | Direct autoregressive signal; lag_168 captures same-hour-last-week |
| Rolling stats | mean/std over 3h, 24h, 168h | Recent trend + volatility without future leakage (shift(1) guard) |
| Weather | cooling_demand, heating_demand, heat_index | Threshold-based load: AC above 24°C, heating below 18°C |
| Economic | gdp_growth, industrial_output, population | Yearly baseline demand level context |

### C. Validation Strategy
- **Chronological hold-out:** all of 2023 as test set; never seen during training or tuning.
- **TimeSeriesSplit (5 folds)** used for cross-validation to prevent leakage during hyperparameter selection.
- **Early stopping** on LightGBM with a 10% validation slice to find optimal `n_estimators`.

### D. Results
*(Fill in after running)*
```
Baseline (lag_1)   MAPE = ?.???%
LightGBM           MAPE = ?.???%
XGBoost            MAPE = ?.???%
Random Forest      MAPE = ?.???%
```

### E. Key Insights from Feature Importance
*(Fill in after running — expected: lag_1 and lag_24 will dominate; hour_sin/cos will rank highly; cooling/heating_demand will outrank raw temperature)*